# 0. Initialiseer - refresh external data en import libraries

### 0.1 De df_dim_sensor tabel bevat de volgende informatie
- **locatie**            = de naam van de locatie waar de sensor is geplaats
- **device_name**        = de naam van de sensor die op de label van de fysieke sensor staat
- **datum_geplaatst**      = de datum waarop de sensor in de grond is geplaatst
- **datum_weggehaald**  = de datum waarop de sensor uit de grond is gehaald. Als deze datum na vandaag ligt, dan zit de sensor daar nog in de grond.
- **diepte_plaatsing**     = de diepte waarop de pinnen van de sensor in de grond zijn geplaats (15cm of 30cm diep)
- **leeftijd_plant**     = de leeftijd van de beplanting waar de de sensor is geplaats (0-1, 1-2 of 2-3 jaar)
- **soort_plant**        = de omschrijving van de beplanting waar de sensor is geplaatst
- **locatie_regio**      = de regio van de stad waar de sensor is geplaatst
- **extra_omschrijving** = een extra omschrijving van de locatie waar de sensor is geplaatst (bijv. in de berm, vlakbij bestrating, ..)
- **old**                = voor start van nieuwe project zijn er al sensoren geplaatst, deze hebben de indicatie 'old'. Van deze data weten we niet hoe betrouwbaar deze is.
- **device_name_org**    = voor het koppelen van de device_id's met de device_name's is de originele (= org) benaming aangepast om de koppeling te vereenvoudigen, daarom voor de zekerheid deze kolom wel behouden.
- **device_id**          = de id van de device zoals deze geregistreerd staat in de quantified portal. Middels de device id's kan de data via de quantified API opgehaald worden.

### 0.2 De df_fact_sensor tabel bevat de volgende informatie
- **gateway_receive_time**  = tijdstip waarop sensor permittivity waarde is ontvangen
- **device**                = device_id waar de permittivity waarde vandaan komt
- **value**                 = de gemeten permittivity waarde

### 0.3 De df_KNMI bevat de volgende informatie
- **YYYYMMDD**  = Datum (YYYY=jaar MM=maand DD=dag)
- **FG**        = Etmaalgemiddelde windsnelheid (in 0.1 m/s)
- (**TG**        = Etmaalgemiddelde temperatuur (in 0.1 graden Celsius)) --> <u>vervangen door Zusterhof</u>
- **SQ**        = Zonneschijnduur (in 0.1 uur) berekend uit de globale straling (-1 voor <0.05 uur)
- **DR**        = Duur van de neerslag (in 0.1 uur)
- (**RH**        = Etmaalsom van de neerslag (in 0.1 mm) (-1 voor <0.05 mm)) --> <u>vervangen door Zusterhof</u>
- **PG**        = Etmaalgemiddelde luchtdruk herleid tot zeeniveau (in 0.1 hPa) berekend uit 24 uurwaarden
- **NG**        = Etmaalgemiddelde bewolking (bedekkingsgraad van de bovenlucht in achtsten, 9=bovenlucht onzichtbaar)
- **UG**        = Etmaalgemiddelde relatieve vochtigheid (in procenten)
- **EV24**      = Referentiegewasverdamping (Makkink) (in 0.1 mm)

### 0.4 De df_weerstation_leiden bevat de volgende informatie
- **datum**         = Datum meetwaarde
- **temperatuur**   = Etmaalgemiddelde temperatuur (in 0.1 graden Celsius)
- **neerslag**      = Etmaalsom van de neerslag (in mm) (-1 voor <0.05 mm)

### Notes
- De temperatuur en neerslag wordt van weerstation **Zusterhof** in Leiden gebruikt. De overige weerdata vanuit het KNMI van weerstation **Schiphol**.
- De KNMI data is ook op uurniveau beschikbaar, maar deze is niet geschikt voor de analyse vanwege datakwaliteit, daarom gebruiken we KNMI data op dagniveau. 


In [ ]:
# TODO:
# Stappenplan van acties voordat dit notebook gedraaid kan worden

In [ ]:
# Import neccesary libraries
from logic_components.data_transformations import resample
import numpy as np
import plotly.express as px
import pandas as pd
from pandas.tseries.offsets import DateOffset

### 0. Refresh & Laad data:
- 1.1 Sensorinformatie vanuit google spreadsheet
- 1.2 Quantified data (= df_fact_sensor)
- 1.3 KNMI data (= df_KNMI)
- 1.4 Weerstation Leiden data (= df_weerstation_leiden)

In [ ]:
# 0. Refresh & Laad data
from api_components import refresh_quantified
from api_components import refresh_external_data
from api_components.data_loading import get_all_data

df_dim_sensor, df_fact_sensor, df_KNMI, df_weerstation_leiden = get_all_data()

### 1. Inspecteer data
- 1.1 Check data info, head and tail
- 1.2 Check data visuals

In [ ]:
# 1.1 Check data info, head and tail

# df_dim_sensor
display(df_dim_sensor.info(5))
display(df_dim_sensor.head(5))
display(df_dim_sensor.tail(5))

# df_fact_sensor
display(df_fact_sensor.info(5))
display(df_fact_sensor.head(5))
display(df_fact_sensor.tail(5))

# df_KNMI
display(df_KNMI.info(5))
display(df_KNMI.head(5))
display(df_KNMI.tail(5))

# df_weerstation_leiden
display(df_weerstation_leiden.info(5))
display(df_weerstation_leiden.head(5))
display(df_weerstation_leiden.tail(5))

# 1.2 Check data visuals
df_visual = df_fact_sensor.copy()
df_visual.sort_values(by=['device', 'gateway_receive_time'], inplace=True)

fig = px.line(df_visual, x='gateway_receive_time', y='value', color='device', title='Time Series of all devices',
              labels={'value': 'Permittivity', 'gateway_receive_time': 'Time'},
              line_group='device', hover_name='device')

fig.show()

cutt_off = df_visual['gateway_receive_time'].max() - DateOffset(months=2)
fig = px.line(df_KNMI[df_KNMI['datum'] > cutt_off], x='datum', y='neerslag', title='Time Series of neerslag KNMI')

fig.show()

1.3 Check latest incoming sensor data

In [ ]:
latest_values = df_fact_sensor.merge(df_dim_sensor, how='left', left_on='device', right_on='device_id')
latest_values = latest_values.loc[latest_values.groupby('device')['gateway_receive_time'].idxmax()]
print(len(latest_values))

latest_values[['device_name','device','gateway_receive_time']].sort_values(by=['gateway_receive_time'], ascending=False)

### 2. Prepareer data
- 2.1 Filter oude locaties er uit
- 2.2 Koppel df_dim_sensor aan df_fact_sensor
- 2.3 Filter per locatie op begin datum plaatsing tot eind datum plaatsing. Locaties zonder einddatum krijgen datum in de toekomst, zodat alle data tot aan vandaag in de dataset zit.
- 2.4 Resample per locatie de tijdreekst op dagniveau. De sensoren geven meerdere sensorwaardes per dag (in principe elke 4 uur), maar dit gaat niet parallel over alle sensoren en voor elke sensor even consequent. Daarom wordt er per dag een gemiddelde, min en max waarde berekend. Daar gaan we de analyse mee doen. De tijdbox functie (add_6H_timebox) gebruiken we momenteel niet, omdat we op dagniveau gaan kijken. Mochten we goede weerdata en sensordata op (4-)uurniveau hebben, dan zouden we ook op een lagere datum_tijd granulariteit analyses kunnen doen.
- 2.5 Prepareer weerdata. De temperatuur en neerslag wordt van weerstation **Zusterhof** in Leiden gebruikt. De overige weerdata vanuit het KNMI van weerstation **Schiphol**. Wanneer de data uit Zusterhof NaN values bevat, wordt deze vervangen door data vanuit het KNMI. **Let op**: we vervangen de negatieve neerslagwaarden (-1) door 0. KNMI maakt onderscheid tussen 'geen regen' (= 0) en 'bijna geen regen' (= -1). In ons geval zien we 'bijna geen' regen als 'geen regen' 
- 2.6 Koppel weerdata aan sensordata
- 2.7 Toevoegen van berekende kolommen

In [ ]:
# 2.1 Filter oude locaties er uit
df_dim_sensor = df_dim_sensor[df_dim_sensor['old'] == 'No']
df_dim_sensor.head()

In [ ]:
# 2.2 Koppel df_dim_sensor aan df_fact_sensor
df = df_dim_sensor.merge(df_fact_sensor, how='left', left_on=[
                         'device_id'], right_on=['device']).drop(columns='device')

In [ ]:
# 2.3 Filter per locatie op begin datum plaatsing tot eind datum plaatsing.

# Filter time series tussen 'datum_geplaatst' en 'datum_opgehaald'
df = df[(df['gateway_receive_time'] > df['datum_geplaatst']) &
        (df['gateway_receive_time'] < df['datum_weggehaald'])]
df.drop(columns=['device_id', 'datum_geplaatst',
        'datum_weggehaald'], inplace=True)

# Rename columns
df.columns = ['locatie', 'device_name', 'soort_plant', 'leeftijd_plant', 'diepte_plaatsing', 'extra_omschrijving', 'locatie_regio',
              'old', 'current_location', 'device_name_org', 'datum_tijd', 'meetwaarde']

# Verzamel locaties
locaties_lst = list(df['locatie'].unique())

# Inspecteer nieuwe dataset
display(df.info(5))
display(df.head(5))
display(df.tail(5))

In [ ]:
# 2.4 Resample per locatie de tijdreekst op dagniveau.

# Resample time series op dagniveau
df_resample = df.set_index(['locatie', 'datum_tijd'])
resample_freq = 'D'  # '6H'

df = resample(df_resample, resample_freq)
df.columns = ['locatie', 'datum', 'min_meetwaarde',
              'max_meetwaarde', 'gem_meetwaarde']

df = df_dim_sensor[df_dim_sensor['locatie'].isin(locaties_lst)].merge(
    df, how='left', left_on=['locatie'], right_on=['locatie'])
df = df[(df['datum'] > df['datum_geplaatst']) &
        (df['datum'] < df['datum_weggehaald'])]

test = df.copy()

df = df[['locatie', 'diepte_plaatsing', 'leeftijd_plant', 'soort_plant', 'locatie_regio',
       'omgevingsfactoren', 'datum','min_meetwaarde', 'max_meetwaarde', 'gem_meetwaarde']]
       
df.head()

In [ ]:
test[test['locatie'] == 'Bachstraat (1)_15']

In [ ]:
# Step 1: Filter df_KNMI to include only data after January 1, 2022
df_weerdata = df_KNMI[df_KNMI['datum'] > '2022-01-01']

# Step 2: Replace NaN values in 'temperatuur' and 'neerslag' columns
df_weerdata['temperatuur'] = np.where(df_weerdata['temperatuur'].isna(), 0, df_weerdata['temperatuur'] / 10)
df_weerdata['neerslag'] = np.where(df_weerdata['neerslag'].isna(), 0, df_weerdata['neerslag'] / 10)

# Step 3: Replace -0.1 values in 'neerslag' column with 0
df_weerdata.loc[df_weerdata['neerslag'] < 0, 'neerslag'] = 0

# Display the modified df_weerdata to user (this would be for testing in an actual use case)
df_weerdata.head()

In [ ]:
#2.6 koppel weerdata aan sensordata
df = df.merge(df_weerdata)
df.head(5)

In [ ]:
df.rename(columns={'omgevingsfactoren': 'extra_omschrijving'}, inplace=True)
df.to_excel('./data/prepped_sensor_data.xlsx', index=False)
df.head(5)

In [ ]:
df_KNMI.to_excel('./data/KNMI.xlsx', index=False)
df_KNMI.head(5)